<a href="https://colab.research.google.com/github/syahidmid/google-colab/blob/main/wordpress/tag/get_tag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import csv
from requests.auth import HTTPBasicAuth

# Jika menggunakan Google Colab
try:
    from google.colab import files, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    userdata = None

# Endpoint Tags
wordpress_url = "https://infojatengpos.com/wp-json/wp/v2/tags"

# Ambil kredensial
if userdata:
    username = userdata.get('syahid_infojateng')
    password = userdata.get('syahid_infojatengpass')
else:
    username = None
    password = None

if not username or not password:
    raise ValueError("Kredensial tidak ditemukan di userdata.")

auth = HTTPBasicAuth(username, password)

# Fungsi ambil semua tags dengan progress
def fetch_all_tags(url, auth):
    all_tags = []
    page = 1

    print("🚀 Memulai pengambilan tag...\n")

    while True:
        print(f"📄 Mengambil halaman {page} ...")

        response = requests.get(
            url,
            auth=auth,
            params={"per_page": 100, "page": page}
        )

        if response.status_code == 200:
            tags = response.json()

            if not tags:
                print("✅ Tidak ada data lagi. Selesai.\n")
                break

            print(f"   ➜ Ditemukan {len(tags)} tag di halaman {page}")

            all_tags.extend(tags)
            page += 1
        else:
            print(f"❌ Gagal di halaman {page}")
            print("Status Code:", response.status_code)
            print(response.text)
            break

    print(f"\n🎯 Total tag berhasil diambil: {len(all_tags)}\n")
    return all_tags


# Ambil semua tag
tags = fetch_all_tags(wordpress_url, auth)

# Simpan ke CSV
if tags:
    csv_filename = "wordpress_tags.csv"

    with open(csv_filename, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)

        # Header
        writer.writerow(["ID", "Name", "Slug", "Count", "Link"])

        for tag in tags:
            writer.writerow([
                tag["id"],
                tag["name"],
                tag["slug"],
                tag["count"],
                tag["link"]
            ])

    print(f"💾 File CSV berhasil disimpan sebagai '{csv_filename}'")

    if IN_COLAB:
        files.download(csv_filename)

else:
    print("⚠ Tidak ada tag ditemukan.")